# Notebook 02: Data Cleaning, Neighborhood Matching, and Spatial Join

**Project:** Predicting Property-Level Housing Violation Risk in Boston
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Purpose

This notebook converts the raw source files into the clean analysis-ready tables that the rest of the pipeline depends on.

It produces four core outputs:

1. `data/processed/violations_clean.csv` - cleaned violation records with parsed dates, canonical neighborhoods, duplicate handling, and spatial flags.
2. `data/processed/properties_clean.csv` - residential parcels with structural fields and a neighborhood label.
3. `data/processed/properties_with_violations.csv` - **property-level modelling table** built by snapping each violation to the nearest residential parcel within 50 m. The classification target `had_violation` lives here.
4. `data/external/neighborhood_structural_features.csv` - GitHub-friendly summary of neighborhood housing-stock context.

The neighborhood matching logic (city label -> point-in-polygon -> ZIP fallback) is shared between violations and properties. **This file is the canonical place for that logic** - notebooks 03 / 04 / 05 just consume the outputs.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'

for d in [RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

violations_path = RAW_DIR / 'violations.csv'
if not violations_path.exists():
    raise FileNotFoundError(f'Missing raw violations file: {violations_path}')

df = pd.read_csv(violations_path, low_memory=False)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head(2)

## 1. Standardize Column Names

We strip whitespace and convert all column names to lowercase snake_case for consistency.

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())

## 2. Parse Dates & Extract Temporal Features

The `status_dttm` column contains datetime strings. We parse it and extract `year`, `month`, and `day_of_week` for temporal analysis.

In [ ]:
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
df['year'] = df['status_dttm'].dt.year
df['month'] = df['status_dttm'].dt.month
df['day_of_week'] = df['status_dttm'].dt.day_name()

print('Records with valid date:', df['status_dttm'].notna().sum())
print('Records missing date:  ', df['status_dttm'].isna().sum())
print('Year range:', df['year'].min(), '-', df['year'].max())

## 3. Neighborhood Assignment

The raw `violation_city` field is not reliable enough by itself. In particular, assigning every generic `Boston` record to `Downtown` creates a false Downtown outlier.

We assign neighborhoods in layers:

1. **Direct city label mapping** for explicit neighborhood names such as `Dorchester`, `Roxbury`, or `East Boston`.
2. **Latitude/longitude polygon lookup** using the official Boston neighborhood GeoJSON boundary file. This is the main fix for generic `Boston` rows.
3. **ZIP fallback** using the dominant neighborhood observed for each ZIP code after the first two assignment methods.

Rows that still cannot be assigned remain flagged with `has_neighborhood = False`; later notebooks keep this as an explicit feature instead of forcing an unreliable neighborhood.

In [ ]:
print('Top 30 raw violation_city values:')
print(df['violation_city'].value_counts(dropna=False).head(30))

# Convert coordinates before spatial assignment.
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Use an explicit canonical neighborhood list so direct mapping does not depend on geometry loading.
canonical_neighborhoods = [
    'Allston', 'Back Bay', 'Beacon Hill', 'Brighton', 'Charlestown', 'Chinatown',
    'Dorchester', 'Downtown', 'East Boston', 'Fenway', 'Hyde Park', 'Jamaica Plain',
    'Longwood', 'Mattapan', 'Mission Hill', 'North End', 'Roslindale', 'Roxbury',
    'South Boston', 'South Boston Waterfront', 'South End', 'West End', 'West Roxbury'
]

precise_city_map = {name.upper(): name for name in canonical_neighborhoods}
precise_city_map.update({
    'E BOSTON': 'East Boston',
    'EASTIE': 'East Boston',
    'SO BOSTON': 'South Boston',
    'SOUTHIE': 'South Boston',
    'JP': 'Jamaica Plain',
    'ROXBURY CROSSING': 'Roxbury',
    'DORCHESTER CENTER': 'Dorchester',
    'CHARLESTOWN': 'Charlestown',
    'EAST BOSTON': 'East Boston',
    'MISSION HILL': 'Mission Hill',
    'MATTAPAN': 'Mattapan',
    'ROXBURY': 'Roxbury',
    'BRIGHTON': 'Brighton',
    'ALLSTON': 'Allston',
    'HYDE PARK': 'Hyde Park',
    'ROSLINDALE': 'Roslindale',
    'JAMAICA PLAIN': 'Jamaica Plain',
    'WEST ROXBURY': 'West Roxbury',
    'SOUTH END': 'South End',
    'FENWAY': 'Fenway',
    'BACK BAY': 'Back Bay',
    'BOSTON/WEST END': 'West End',
    'FINANCIAL DISTRICT': 'Downtown',
    'BAY VILLAGE': 'Downtown',
    'LEATHER DISTRICT': 'Downtown',
    'NORTHEND': 'North End',
    'NORTH END': 'North End',
})

# Normalize raw city labels. Remove trailing slashes so values like 'Dorchester/' map correctly.
city_clean = (
    df['violation_city']
    .astype('string')
    .str.strip()
    .str.upper()
    .str.replace(r'/+$', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

# Generic Boston labels are too broad and must be assigned by coordinates or ZIP, not by direct city label.
generic_city_labels = {'BOSTON', 'CITY OF BOSTON'}
df['neighborhood_from_city'] = city_clean.map(precise_city_map)
df.loc[city_clean.isin(generic_city_labels), 'neighborhood_from_city'] = pd.NA

# Spatial polygon assignment from the official Boston neighborhood GeoJSON boundary file.
# This uses plain Python point-in-polygon logic so the notebook does not depend on shapely being installed.
import json

boundary_geojson_path = EXTERNAL_DIR / 'boston_neighborhood_boundaries.geojson'
df['neighborhood_from_spatial'] = pd.NA

def iter_coordinate_pairs(coordinates):
    if not coordinates:
        return
    first = coordinates[0]
    if isinstance(first, (int, float)):
        yield coordinates
    else:
        for item in coordinates:
            yield from iter_coordinate_pairs(item)

def geometry_bbox(geometry):
    points = list(iter_coordinate_pairs(geometry.get('coordinates', [])))
    if not points:
        return None
    lons = [float(point[0]) for point in points]
    lats = [float(point[1]) for point in points]
    return min(lons), min(lats), max(lons), max(lats)

def point_on_segment(lon, lat, x1, y1, x2, y2, eps=1e-10):
    cross = (lat - y1) * (x2 - x1) - (lon - x1) * (y2 - y1)
    if abs(cross) > eps:
        return False
    return min(x1, x2) - eps <= lon <= max(x1, x2) + eps and min(y1, y2) - eps <= lat <= max(y1, y2) + eps

def point_in_ring(lon, lat, ring):
    if len(ring) < 3:
        return False
    inside = False
    for i in range(len(ring)):
        x1, y1 = float(ring[i][0]), float(ring[i][1])
        x2, y2 = float(ring[(i + 1) % len(ring)][0]), float(ring[(i + 1) % len(ring)][1])
        if point_on_segment(lon, lat, x1, y1, x2, y2):
            return True
        if (y1 > lat) != (y2 > lat):
            x_intersect = (x2 - x1) * (lat - y1) / (y2 - y1) + x1
            if lon <= x_intersect:
                inside = not inside
    return inside

def point_in_polygon(lon, lat, polygon_coordinates):
    if not polygon_coordinates or not point_in_ring(lon, lat, polygon_coordinates[0]):
        return False
    for hole in polygon_coordinates[1:]:
        if point_in_ring(lon, lat, hole):
            return False
    return True

def geometry_contains_point(lon, lat, geometry):
    geometry_type = geometry.get('type')
    coordinates = geometry.get('coordinates', [])
    if geometry_type == 'Polygon':
        return point_in_polygon(lon, lat, coordinates)
    if geometry_type == 'MultiPolygon':
        return any(point_in_polygon(lon, lat, polygon) for polygon in coordinates)
    return False

if boundary_geojson_path.exists():
    with open(boundary_geojson_path, encoding='utf-8') as file:
        boundary_geojson = json.load(file)

    # Keep the neighborhood set compatible with the population table used later for per-1,000 rates.
    boundary_name_map = {
        'Bay Village': 'Downtown',
        'Leather District': 'Downtown',
        'Harbor Islands': pd.NA,
    }
    boundary_records = []
    for feature in boundary_geojson.get('features', []):
        raw_name = feature.get('properties', {}).get('name')
        neighborhood_name = boundary_name_map.get(raw_name, raw_name)
        geometry = feature.get('geometry')
        bbox = geometry_bbox(geometry) if geometry else None
        if pd.notna(neighborhood_name) and neighborhood_name in canonical_neighborhoods and geometry and bbox:
            boundary_records.append((neighborhood_name, geometry, bbox))

    def assign_neighborhood_from_point(lat, lon):
        if pd.isna(lat) or pd.isna(lon):
            return pd.NA
        lat = float(lat)
        lon = float(lon)
        for neighborhood_name, geometry, bbox in boundary_records:
            min_lon, min_lat, max_lon, max_lat = bbox
            if min_lon <= lon <= max_lon and min_lat <= lat <= max_lat:
                if geometry_contains_point(lon, lat, geometry):
                    return neighborhood_name
        return pd.NA

    coord_mask = df['latitude'].notna() & df['longitude'].notna()
    df.loc[coord_mask, 'neighborhood_from_spatial'] = [
        assign_neighborhood_from_point(lat, lon)
        for lat, lon in zip(df.loc[coord_mask, 'latitude'], df.loc[coord_mask, 'longitude'])
    ]
    print(f'Loaded {len(boundary_records)} neighborhood boundary polygons from {boundary_geojson_path.name}.')
    print('Spatial polygon assignment completed.')
else:
    print(f'Neighborhood boundary GeoJSON not found: {boundary_geojson_path}')

# ZIP fallback learned from rows already assigned by explicit city label or spatial polygon.
df['zip5'] = df['violation_zip'].astype('string').str.extract(r'(\d{5})')[0]
zip_training = df.copy()
zip_training['zip_training_neighborhood'] = zip_training['neighborhood_from_spatial'].fillna(zip_training['neighborhood_from_city'])
zip_counts = (
    zip_training.dropna(subset=['zip5', 'zip_training_neighborhood'])
    .groupby(['zip5', 'zip_training_neighborhood'])
    .size()
    .reset_index(name='n')
)
zip_mode = (
    zip_counts.sort_values(['zip5', 'n'], ascending=[True, False])
    .drop_duplicates('zip5')
    .set_index('zip5')['zip_training_neighborhood']
    .to_dict()
)
# Manual ZIP fallback for generic Boston rows and ZIPs with weak learned support.
# ZIP codes do not perfectly match neighborhoods, so these are conservative approximations used only after city/spatial assignment.
manual_zip_map = {
    '02108': 'Beacon Hill',
    '02109': 'Downtown',
    '02110': 'Downtown',
    '02111': 'Chinatown',
    '02113': 'North End',
    '02114': 'West End',
    '02115': 'Fenway',
    '02116': 'Back Bay',
    '02118': 'South End',
    '02119': 'Roxbury',
    '02120': 'Mission Hill',
    '02121': 'Dorchester',
    '02122': 'Dorchester',
    '02124': 'Dorchester',
    '02125': 'Dorchester',
    '02126': 'Mattapan',
    '02127': 'South Boston',
    '02128': 'East Boston',
    '02129': 'Charlestown',
    '02130': 'Jamaica Plain',
    '02131': 'Roslindale',
    '02132': 'West Roxbury',
    '02134': 'Allston',
    '02135': 'Brighton',
    '02136': 'Hyde Park',
    '02163': 'Allston',
    '02199': 'Back Bay',
    '02201': 'Downtown',
    '02203': 'Downtown',
    '02210': 'South Boston Waterfront',
    '02215': 'Fenway',
}

df['neighborhood_from_zip_manual'] = df['zip5'].map(manual_zip_map)
df['neighborhood_from_zip_learned'] = df['zip5'].map(zip_mode)
df['neighborhood_from_zip'] = df['neighborhood_from_zip_manual'].fillna(df['neighborhood_from_zip_learned'])

# Final assignment. For explicit labels we trust city labels; generic Boston uses spatial/ZIP because city label was set to NA.
df['neighborhood'] = pd.NA
df['neighborhood_assignment_method'] = pd.NA
for source_col, method_name in [
    ('neighborhood_from_city', 'city_label'),
    ('neighborhood_from_spatial', 'lat_lon_polygon'),
    ('neighborhood_from_zip', 'zip_dominant'),
]:
    mask = df['neighborhood'].isna() & df[source_col].notna()
    df.loc[mask, 'neighborhood'] = df.loc[mask, source_col]
    df.loc[mask, 'neighborhood_assignment_method'] = method_name

mapped = df['neighborhood'].notna().sum()
total = len(df)
print(f'Mapped to neighborhood: {mapped:,} / {total:,} ({100*mapped/total:.1f}%)')
print('\nAssignment method counts:')
print(df['neighborhood_assignment_method'].value_counts(dropna=False).to_string())

final_neighborhood_counts = (
    df['neighborhood']
    .value_counts(dropna=False)
    .rename_axis('neighborhood')
    .reset_index(name='final_row_count')
)
print('\nFinal neighborhood counts after reassignment:')
print(final_neighborhood_counts.to_string(index=False))

raw_boston_mask = city_clean.isin(generic_city_labels)
boston_assignment = df.loc[
    raw_boston_mask,
    ['violation_city', 'neighborhood', 'neighborhood_assignment_method', 'violation_zip', 'latitude', 'longitude']
].copy()
boston_reassignment_counts = (
    boston_assignment['neighborhood']
    .value_counts(dropna=False)
    .rename_axis('assigned_neighborhood')
    .reset_index(name='raw_boston_row_count')
)
print('\nRaw Boston-labeled rows redistributed across neighborhoods:')
print(boston_reassignment_counts.to_string(index=False))
print('\nAssignment methods for raw Boston-labeled rows:')
print(boston_assignment['neighborhood_assignment_method'].value_counts(dropna=False).to_string())
print(
    f"\nRaw Boston-labeled rows mapped to Downtown: "
    f"{(boston_assignment['neighborhood'] == 'Downtown').sum():,} / {len(boston_assignment):,}"
)
display(boston_assignment.head(10))

unmapped_city_counts = df.loc[df['neighborhood'].isna(), 'violation_city'].value_counts(dropna=False)
print('\nRemaining unmapped raw city values:')
print(unmapped_city_counts.head(30).to_string())

## 4. Select Relevant Columns

We keep the cleaned neighborhood assignment plus the fields needed for downstream EDA, normalization, and modeling. Address components are retained for auditability, but they are not used directly as model features.

In [ ]:
keep_cols = [
    'case_no', 'status', 'status_dttm', 'year', 'month', 'day_of_week',
    'code', 'description',
    'violation_city', 'neighborhood', 'neighborhood_assignment_method',
    'violation_stno', 'violation_sthigh', 'violation_street', 'violation_suffix', 'violation_zip', 'ward',
    'latitude', 'longitude',
]
df_clean = df[[c for c in keep_cols if c in df.columns]].copy()
print(f'Kept columns: {df_clean.columns.tolist()}')
print(f'Shape after column selection: {df_clean.shape}')

## 5. Handle Missing Values

- Records missing `status_dttm` retain their other fields and are included in non-temporal analyses.
- `description` NaN values are filled with `'Unknown'`.
- Records that still lack a neighborhood after city, spatial, and ZIP assignment are flagged with `has_neighborhood = False`; the classification notebook can use this flag without inventing a neighborhood.

In [ ]:
df_clean['description'] = df_clean['description'].fillna('Unknown')
df_clean['code'] = df_clean['code'].astype(str).str.strip()

# Convert lat/lon to numeric again after column selection.
df_clean['latitude'] = pd.to_numeric(df_clean['latitude'], errors='coerce')
df_clean['longitude'] = pd.to_numeric(df_clean['longitude'], errors='coerce')

# Flag records that have valid spatial and neighborhood data.
df_clean['has_coords'] = df_clean['latitude'].notna() & df_clean['longitude'].notna()
df_clean['has_neighborhood'] = df_clean['neighborhood'].notna()

print('Records with coordinates:', df_clean['has_coords'].sum())
print('Records with neighborhood:', df_clean['has_neighborhood'].sum())
print('Records still missing neighborhood:', (~df_clean['has_neighborhood']).sum())
print('\nNeighborhood assignment methods in cleaned data:')
print(df_clean['neighborhood_assignment_method'].value_counts(dropna=False))

## 6. Remove Duplicates

We deduplicate on `case_no`, keeping the most recent record per case.

In [ ]:
before = len(df_clean)
df_clean = df_clean.sort_values('status_dttm', ascending=False)
df_clean = df_clean.drop_duplicates(subset='case_no', keep='first').reset_index(drop=True)
after = len(df_clean)
print(f'Removed {before - after:,} duplicate case numbers. Remaining: {after:,}')

## 7. Final Quality Check & Save

We verify the cleaned dataset and write it to `data/processed/violations_clean.csv`.

In [ ]:
print('=== Final Cleaned Dataset ===' )
print(f'Shape: {df_clean.shape}')
print(f'Date range: {df_clean["status_dttm"].min()} -> {df_clean["status_dttm"].max()}')
print(f'Unique neighborhoods: {df_clean["neighborhood"].nunique()}')
print(f'Unique violation codes: {df_clean["code"].nunique()}')
print()
print('Missing values per column:')
print(df_clean.isnull().sum())

In [ ]:
out_path = PROCESSED_DIR / 'violations_clean.csv'
df_clean.to_csv(out_path, index=False)
print(f'Saved cleaned data -> {out_path}')
df_clean.head(5)

## 8. Clean the Property Assessment File

The property assessment file has ~182k parcel rows. The pipeline produces two outputs from it:

1. **Parcel-level cleaned table** (`data/processed/properties_clean.csv`) - one row per residential parcel, with derived structural fields and a neighborhood label assigned by the same three-layer logic used for violations (city label, polygon match, ZIP fallback). This is the modelling table.
2. **Neighborhood structural aggregates** (`data/external/neighborhood_structural_features.csv`) - small lookup table used as contextual features later.

Cleaning decisions:

- Keep fields related to housing stock: `YR_BUILT`, `YR_REMODEL`, `TOTAL_VALUE`, `BLDG_VALUE`, `LAND_VALUE`, `LIVING_AREA`, `GROSS_AREA`, `LAND_SF`, `RES_UNITS`, `COM_UNITS`, `NUM_BLDGS`, `BED_RMS`, `FULL_BTH`, `HLF_BTH`, `BLDG_TYPE`, `STRUCTURE_CLASS`, `OVERALL_COND`, `INT_COND`, `EXT_COND`, `HEAT_TYPE`, `OWN_OCC`, `LU`, `LU_DESC`.
- Convert currency and numeric strings into numeric values; treat impossible build years outside `1700-2025` as missing.
- Identify residential parcels using `LU` codes and `LU_DESC` text matches.
- Assign neighborhoods using the same three-layer logic as violations: explicit city label, point-in-polygon match against the Boston neighborhood GeoJSON, and ZIP fallback.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Make this cell safe to run by itself.
if 'PROJECT_ROOT' not in globals():
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent

RAW_DIR = globals().get('RAW_DIR', PROJECT_ROOT / 'data' / 'raw')
PROCESSED_DIR = globals().get('PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed')
EXTERNAL_DIR = globals().get('EXTERNAL_DIR', PROJECT_ROOT / 'data' / 'external')

for d in [RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if 'df_clean' not in globals():
    clean_path = PROCESSED_DIR / 'violations_clean.csv'
    if clean_path.exists():
        df_clean = pd.read_csv(clean_path, low_memory=False)
        print(f'Loaded cleaned violations for ZIP/neighborhood mapping: {clean_path}')
    else:
        raise NameError(
            'df_clean is not defined and data/processed/violations_clean.csv does not exist. '
            'Run the earlier cleaning cells first, then rerun this cell.'
        )

property_path = EXTERNAL_DIR / 'boston_property_assessment_data.csv'
properties_clean_path = PROCESSED_DIR / 'properties_clean.csv'
structural_external_path = EXTERNAL_DIR / 'neighborhood_structural_features.csv'
structural_processed_path = PROCESSED_DIR / 'neighborhood_structural_features.csv'
summary_path = PROCESSED_DIR / 'property_assessment_feature_summary.csv'

if not property_path.exists():
    raise FileNotFoundError(
        f'Missing property assessment CSV at {property_path}. '
        'Download boston_property_assessment_data.csv from data.boston.gov and place it in data/external/.'
    )

# Local copies of the polygon helpers so this cell does not depend on cell ordering.
def _iter_coordinate_pairs(coordinates):
    if not coordinates:
        return
    first = coordinates[0]
    if isinstance(first, (int, float)):
        yield coordinates
    else:
        for item in coordinates:
            yield from _iter_coordinate_pairs(item)

def _geometry_bbox(geometry):
    points = list(_iter_coordinate_pairs(geometry.get('coordinates', [])))
    if not points:
        return None
    lons = [float(point[0]) for point in points]
    lats = [float(point[1]) for point in points]
    return min(lons), min(lats), max(lons), max(lats)

def _point_on_segment(lon, lat, x1, y1, x2, y2, eps=1e-10):
    cross = (lat - y1) * (x2 - x1) - (lon - x1) * (y2 - y1)
    if abs(cross) > eps:
        return False
    return min(x1, x2) - eps <= lon <= max(x1, x2) + eps and min(y1, y2) - eps <= lat <= max(y1, y2) + eps

def _point_in_ring(lon, lat, ring):
    if len(ring) < 3:
        return False
    inside = False
    for i in range(len(ring)):
        x1, y1 = float(ring[i][0]), float(ring[i][1])
        x2, y2 = float(ring[(i + 1) % len(ring)][0]), float(ring[(i + 1) % len(ring)][1])
        if _point_on_segment(lon, lat, x1, y1, x2, y2):
            return True
        if (y1 > lat) != (y2 > lat):
            x_intersect = (x2 - x1) * (lat - y1) / (y2 - y1) + x1
            if lon <= x_intersect:
                inside = not inside
    return inside

def _point_in_polygon(lon, lat, polygon_coordinates):
    if not polygon_coordinates or not _point_in_ring(lon, lat, polygon_coordinates[0]):
        return False
    for hole in polygon_coordinates[1:]:
        if _point_in_ring(lon, lat, hole):
            return False
    return True

def _geometry_contains_point(lon, lat, geometry):
    geometry_type = geometry.get('type')
    coordinates = geometry.get('coordinates', [])
    if geometry_type == 'Polygon':
        return _point_in_polygon(lon, lat, coordinates)
    if geometry_type == 'MultiPolygon':
        return any(_point_in_polygon(lon, lat, polygon) for polygon in coordinates)
    return False

usecols = [
    'PID', 'CITY', 'LU', 'LU_DESC', 'YR_BUILT', 'YR_REMODEL',
    'TOTAL_VALUE', 'BLDG_VALUE', 'LAND_VALUE', 'LIVING_AREA',
    'GROSS_AREA', 'LAND_SF', 'RES_UNITS', 'COM_UNITS', 'NUM_BLDGS',
    'BED_RMS', 'FULL_BTH', 'HLF_BTH', 'KITCHENS', 'FIREPLACES',
    'BLDG_TYPE', 'STRUCTURE_CLASS', 'OVERALL_COND', 'INT_COND', 'EXT_COND',
    'HEAT_TYPE', 'AC_TYPE', 'OWN_OCC',
    'ZIP_CODE', 'geocode_latitude', 'geocode_longitude',
    'ST_NUM', 'ST_NAME',
]
prop_raw = pd.read_csv(property_path, usecols=usecols, low_memory=False)
print(f'Loaded {len(prop_raw):,} parcel rows from property assessment file.')
prop = prop_raw.copy()

def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r'[$,]', '', regex=True).str.strip(),
        errors='coerce'
    )

numeric_cols = [
    'TOTAL_VALUE', 'BLDG_VALUE', 'LAND_VALUE', 'LIVING_AREA', 'GROSS_AREA',
    'LAND_SF', 'RES_UNITS', 'COM_UNITS', 'NUM_BLDGS',
    'BED_RMS', 'FULL_BTH', 'HLF_BTH', 'KITCHENS', 'FIREPLACES',
    'YR_BUILT', 'YR_REMODEL'
]
for col in numeric_cols:
    if col in prop.columns:
        prop[col] = clean_numeric(prop[col])

prop['geocode_latitude'] = pd.to_numeric(prop['geocode_latitude'], errors='coerce')
prop['geocode_longitude'] = pd.to_numeric(prop['geocode_longitude'], errors='coerce')

# Derived structural features.
valid_year = prop['YR_BUILT'].between(1700, 2025)
prop.loc[~valid_year, 'YR_BUILT'] = np.nan
prop['building_age_2025'] = 2025 - prop['YR_BUILT']
prop['was_remodeled'] = prop['YR_REMODEL'].between(1700, 2025).astype(int)
prop['is_pre_1940'] = (prop['YR_BUILT'] < 1940).astype('Int64')
prop['is_pre_1900'] = (prop['YR_BUILT'] < 1900).astype('Int64')

lu = prop['LU'].astype(str).str.upper().str.strip()
lu_desc = prop['LU_DESC'].astype(str).str.upper()
prop['is_residential'] = lu.isin(['R1', 'R2', 'R3', 'R4', 'A', 'CD', 'CM', 'RC']) | lu_desc.str.contains('RES|DWELLING|CONDO|APT|HOUSING', na=False)
prop['is_condo'] = (lu.isin(['CD', 'CM']) | lu_desc.str.contains('CONDO', na=False)).astype(int)
prop['is_multifamily'] = (lu.isin(['R2', 'R3', 'R4', 'A']) | lu_desc.str.contains('TWO-FAM|THREE-FAM|APT|MULTI|HOUSING', na=False)).astype(int)

# Same three-layer neighborhood assignment as for violations: city label, polygon match, ZIP fallback.
canonical_neighborhoods = [
    'Allston', 'Back Bay', 'Beacon Hill', 'Brighton', 'Charlestown', 'Chinatown',
    'Dorchester', 'Downtown', 'East Boston', 'Fenway', 'Hyde Park', 'Jamaica Plain',
    'Longwood', 'Mattapan', 'Mission Hill', 'North End', 'Roslindale', 'Roxbury',
    'South Boston', 'South Boston Waterfront', 'South End', 'West End', 'West Roxbury'
]
city_map = {name.upper(): name for name in canonical_neighborhoods}
city_map.update({
    'DORCHESTER CENTER': 'Dorchester',
    'ROXBURY CROSSING': 'Roxbury',
    'EASTIE': 'East Boston',
    'JAMAICA PLAIN': 'Jamaica Plain',
})
prop['city_clean'] = prop['CITY'].astype(str).str.strip().str.upper()
prop['zip5'] = prop['ZIP_CODE'].astype(str).str.extract(r'(\d{4,5})')[0].str.zfill(5)
prop['neighborhood_from_city'] = prop['city_clean'].map(city_map)
generic_city_labels = {'BOSTON', 'CITY OF BOSTON', '', 'NAN'}
prop.loc[prop['city_clean'].isin(generic_city_labels), 'neighborhood_from_city'] = pd.NA

boundary_geojson_path = EXTERNAL_DIR / 'boston_neighborhood_boundaries.geojson'
prop['neighborhood_from_spatial'] = pd.NA
property_boundary_records = []
if boundary_geojson_path.exists():
    with open(boundary_geojson_path, encoding='utf-8') as file:
        boundary_geojson = json.load(file)
    boundary_name_map = {
        'Bay Village': 'Downtown',
        'Leather District': 'Downtown',
        'Harbor Islands': pd.NA,
    }
    for feature in boundary_geojson.get('features', []):
        raw_name = feature.get('properties', {}).get('name')
        neighborhood_name = boundary_name_map.get(raw_name, raw_name)
        geometry = feature.get('geometry')
        bbox = _geometry_bbox(geometry) if geometry else None
        if pd.notna(neighborhood_name) and neighborhood_name in canonical_neighborhoods and geometry and bbox:
            property_boundary_records.append((neighborhood_name, geometry, bbox))

def assign_property_neighborhood(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return pd.NA
    lat = float(lat)
    lon = float(lon)
    for neighborhood_name, geometry, bbox in property_boundary_records:
        min_lon, min_lat, max_lon, max_lat = bbox
        if min_lon <= lon <= max_lon and min_lat <= lat <= max_lat:
            if _geometry_contains_point(lon, lat, geometry):
                return neighborhood_name
    return pd.NA

coord_mask = prop['geocode_latitude'].notna() & prop['geocode_longitude'].notna()
prop.loc[coord_mask, 'neighborhood_from_spatial'] = [
    assign_property_neighborhood(lat, lon)
    for lat, lon in zip(prop.loc[coord_mask, 'geocode_latitude'], prop.loc[coord_mask, 'geocode_longitude'])
]

manual_zip_map = {
    '02108': 'Beacon Hill', '02109': 'Downtown', '02110': 'Downtown', '02111': 'Chinatown',
    '02113': 'North End', '02114': 'West End', '02115': 'Fenway', '02116': 'Back Bay',
    '02118': 'South End', '02119': 'Roxbury', '02120': 'Mission Hill', '02121': 'Dorchester',
    '02122': 'Dorchester', '02124': 'Dorchester', '02125': 'Dorchester', '02126': 'Mattapan',
    '02127': 'South Boston', '02128': 'East Boston', '02129': 'Charlestown', '02130': 'Jamaica Plain',
    '02131': 'Roslindale', '02132': 'West Roxbury', '02134': 'Allston', '02135': 'Brighton',
    '02136': 'Hyde Park', '02163': 'Allston', '02199': 'Back Bay', '02201': 'Downtown',
    '02203': 'Downtown', '02210': 'South Boston Waterfront', '02215': 'Fenway',
}
prop['neighborhood_from_zip'] = prop['zip5'].map(manual_zip_map)

prop['neighborhood'] = pd.NA
prop['neighborhood_assignment_method'] = pd.NA
for source_col, method_name in [
    ('neighborhood_from_city', 'city_label'),
    ('neighborhood_from_spatial', 'lat_lon_polygon'),
    ('neighborhood_from_zip', 'zip_dominant'),
]:
    mask = prop['neighborhood'].isna() & prop[source_col].notna()
    prop.loc[mask, 'neighborhood'] = prop.loc[mask, source_col]
    prop.loc[mask, 'neighborhood_assignment_method'] = method_name

mapped_props = prop['neighborhood'].notna().sum()
print(f'Properties mapped to a neighborhood: {mapped_props:,} / {len(prop):,} ({100*mapped_props/len(prop):.1f}%)')
print('Property neighborhood method counts:')
print(prop['neighborhood_assignment_method'].value_counts(dropna=False).to_string())

residential = prop[prop['is_residential'] & prop['geocode_latitude'].notna() & prop['geocode_longitude'].notna()].copy()
residential = residential[residential['neighborhood'].notna()].copy()
print(f'Residential parcels with coords + neighborhood: {len(residential):,}')

parcel_keep_cols = [
    'PID', 'neighborhood', 'neighborhood_assignment_method', 'zip5', 'CITY',
    'ST_NUM', 'ST_NAME', 'geocode_latitude', 'geocode_longitude',
    'LU', 'LU_DESC', 'BLDG_TYPE', 'STRUCTURE_CLASS',
    'YR_BUILT', 'YR_REMODEL', 'building_age_2025', 'was_remodeled', 'is_pre_1940', 'is_pre_1900',
    'TOTAL_VALUE', 'BLDG_VALUE', 'LAND_VALUE',
    'LIVING_AREA', 'GROSS_AREA', 'LAND_SF',
    'RES_UNITS', 'COM_UNITS', 'NUM_BLDGS',
    'BED_RMS', 'FULL_BTH', 'HLF_BTH', 'KITCHENS', 'FIREPLACES',
    'OVERALL_COND', 'INT_COND', 'EXT_COND',
    'HEAT_TYPE', 'AC_TYPE', 'OWN_OCC',
    'is_residential', 'is_condo', 'is_multifamily',
]
parcel_keep_cols = [c for c in parcel_keep_cols if c in residential.columns]
residential[parcel_keep_cols].to_csv(properties_clean_path, index=False)
print(f'Saved parcel-level cleaned table -> {properties_clean_path}')

structural = residential.groupby('neighborhood').agg(
    property_count=('LU', 'size'),
    median_building_age=('building_age_2025', 'median'),
    mean_building_age=('building_age_2025', 'mean'),
    share_pre_1940=('is_pre_1940', 'mean'),
    share_pre_1900=('is_pre_1900', 'mean'),
    share_remodeled=('was_remodeled', 'mean'),
    median_total_value=('TOTAL_VALUE', 'median'),
    median_building_value=('BLDG_VALUE', 'median'),
    median_land_value=('LAND_VALUE', 'median'),
    median_living_area=('LIVING_AREA', 'median'),
    median_gross_area=('GROSS_AREA', 'median'),
    median_land_sf=('LAND_SF', 'median'),
    share_condo=('is_condo', 'mean'),
    share_multifamily=('is_multifamily', 'mean'),
).reset_index()

population_path = EXTERNAL_DIR / 'neighborhood_population.csv'
if not population_path.exists():
    raise FileNotFoundError(f'Missing population table: {population_path}. Run Notebook 01 first.')
pop = pd.read_csv(population_path)[['neighborhood', 'population_2025']]
structural = structural.merge(pop, on='neighborhood', how='left')
structural['properties_per_1000_residents'] = structural['property_count'] / structural['population_2025'] * 1000

round_cols = [c for c in structural.columns if c not in ['neighborhood', 'property_count', 'population_2025']]
structural[round_cols] = structural[round_cols].round(3)
structural = structural.sort_values('neighborhood').reset_index(drop=True)
structural.to_csv(structural_processed_path, index=False)
structural.to_csv(structural_external_path, index=False)
print(f'Saved structural aggregates -> {structural_processed_path}')
print(f'Saved GitHub-friendly copy   -> {structural_external_path}')

feature_summary = pd.DataFrame({
    'metric': [
        'raw_property_rows', 'rows_after_residential_filter',
        'rows_with_neighborhood_and_coords', 'neighborhoods_output',
        'missing_year_built_pct_residential', 'missing_total_value_pct_residential'
    ],
    'value': [
        len(prop_raw), int(prop['is_residential'].sum()),
        len(residential), structural['neighborhood'].nunique(),
        round(residential['YR_BUILT'].isna().mean() * 100, 2),
        round(residential['TOTAL_VALUE'].isna().mean() * 100, 2),
    ]
})
feature_summary.to_csv(summary_path, index=False)
display(structural.head())
display(residential[parcel_keep_cols].head(3))


## 9. Spatial Join: Snap Each Violation to the Nearest Property

This is the step that turns the project from neighborhood-level (~24 rows) into property-level (~150,000 rows).

For every violation record with valid `latitude`/`longitude`, we find the nearest residential parcel using a haversine `BallTree`. If the nearest parcel is within **50 m**, that violation is attributed to that property's `PID`. Each property then gets:

- `had_violation` - 1 if any violation was snapped to it, else 0 (this is the modelling target).
- `violation_count` - total snapped violations (auditing only; not used as a feature because it leaks the target).
- `recent_violation_count` - violations with `status_dttm` >= 2022-01-01 (auditing only).

The 50 m radius was chosen by inspection: rooftop-level geocoding for both files clusters at the building, and 50 m is wide enough to absorb GPS noise but tight enough that adjacent parcels do not absorb each other's violations.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.neighbors import BallTree

# Make this cell safe to run by itself.
if 'PROJECT_ROOT' not in globals():
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent
PROCESSED_DIR = globals().get('PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed')

violations_clean_path = PROCESSED_DIR / 'violations_clean.csv'
properties_clean_path = PROCESSED_DIR / 'properties_clean.csv'
properties_with_violations_path = PROCESSED_DIR / 'properties_with_violations.csv'

if 'df_clean' not in globals():
    df_clean = pd.read_csv(violations_clean_path, low_memory=False)
properties_df = pd.read_csv(properties_clean_path, low_memory=False)

violations_geo = df_clean.copy()
violations_geo['latitude'] = pd.to_numeric(violations_geo['latitude'], errors='coerce')
violations_geo['longitude'] = pd.to_numeric(violations_geo['longitude'], errors='coerce')
violations_geo = violations_geo[violations_geo['latitude'].between(42.22, 42.40) & violations_geo['longitude'].between(-71.19, -70.99)].copy()
violations_geo = violations_geo.dropna(subset=['latitude', 'longitude']).copy()
violations_geo['status_dttm'] = pd.to_datetime(violations_geo['status_dttm'], errors='coerce')

properties_df['geocode_latitude'] = pd.to_numeric(properties_df['geocode_latitude'], errors='coerce')
properties_df['geocode_longitude'] = pd.to_numeric(properties_df['geocode_longitude'], errors='coerce')
properties_geo = properties_df.dropna(subset=['geocode_latitude', 'geocode_longitude']).copy().reset_index(drop=True)

# BallTree with haversine expects radians. Earth radius is 6,371,000 m, so 50 m = 50/6_371_000 radians.
property_coords_rad = np.radians(properties_geo[['geocode_latitude', 'geocode_longitude']].to_numpy())
violation_coords_rad = np.radians(violations_geo[['latitude', 'longitude']].to_numpy())

print(f'Building BallTree over {len(properties_geo):,} parcels...')
tree = BallTree(property_coords_rad, metric='haversine')
distances_rad, indices = tree.query(violation_coords_rad, k=1)
distances_m = distances_rad[:, 0] * 6_371_000
nearest_property_index = indices[:, 0]

snap_radius_m = 50.0
within_radius = distances_m <= snap_radius_m

violations_geo = violations_geo.reset_index(drop=True)
violations_geo['snapped_property_idx'] = np.where(within_radius, nearest_property_index, -1)
violations_geo['snapped_distance_m'] = distances_m
violations_geo['snapped_PID'] = np.where(
    within_radius,
    properties_geo.loc[nearest_property_index, 'PID'].to_numpy(),
    np.nan,
)

snap_summary = pd.DataFrame({
    'metric': [
        'violations_with_coords', 'violations_snapped_within_50m',
        'snap_rate_pct', 'median_snap_distance_m', 'mean_snap_distance_m'
    ],
    'value': [
        len(violations_geo),
        int(within_radius.sum()),
        round(within_radius.mean() * 100, 2),
        round(float(np.median(distances_m[within_radius])), 2) if within_radius.any() else np.nan,
        round(float(np.mean(distances_m[within_radius])), 2) if within_radius.any() else np.nan,
    ]
})
snap_summary.to_csv(PROCESSED_DIR / 'spatial_join_summary.csv', index=False)
print(snap_summary.to_string(index=False))
violations_geo[['case_no', 'snapped_PID', 'snapped_distance_m']].head()


## 10. Build the Property-Level Modelling Table

Aggregate the snapped violations to one row per property. Every residential parcel from `properties_clean.csv` is kept, even if it has zero snapped violations - those become the negative class. The output `properties_with_violations.csv` is the row source for notebooks 03, 04, and 05.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    current_dir = Path.cwd().resolve()
    PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent
PROCESSED_DIR = globals().get('PROCESSED_DIR', PROJECT_ROOT / 'data' / 'processed')

properties_clean_path = PROCESSED_DIR / 'properties_clean.csv'
properties_with_violations_path = PROCESSED_DIR / 'properties_with_violations.csv'

if 'properties_df' not in globals():
    properties_df = pd.read_csv(properties_clean_path, low_memory=False)
if 'violations_geo' not in globals():
    raise NameError('Run the spatial-join cell above first.')

snapped = violations_geo.dropna(subset=['snapped_PID']).copy()
snapped['snapped_PID'] = snapped['snapped_PID'].astype(str)

snapped['recent_violation'] = (snapped['status_dttm'] >= pd.Timestamp('2022-01-01')).astype(int)

per_property = snapped.groupby('snapped_PID').agg(
    violation_count=('case_no', 'count'),
    recent_violation_count=('recent_violation', 'sum'),
    last_violation_date=('status_dttm', 'max'),
    first_violation_date=('status_dttm', 'min'),
).reset_index().rename(columns={'snapped_PID': 'PID'})

properties_df['PID'] = properties_df['PID'].astype(str)
property_target = properties_df.merge(per_property, on='PID', how='left')
property_target['violation_count'] = property_target['violation_count'].fillna(0).astype(int)
property_target['recent_violation_count'] = property_target['recent_violation_count'].fillna(0).astype(int)
property_target['had_violation'] = (property_target['violation_count'] >= 1).astype(int)
property_target['had_recent_violation'] = (property_target['recent_violation_count'] >= 1).astype(int)

property_target.to_csv(properties_with_violations_path, index=False)

target_summary = pd.DataFrame({
    'metric': [
        'total_properties', 'properties_with_violation', 'positive_class_rate_pct',
        'properties_with_recent_violation', 'recent_positive_rate_pct', 'mean_violations_per_positive_property',
    ],
    'value': [
        len(property_target),
        int(property_target['had_violation'].sum()),
        round(property_target['had_violation'].mean() * 100, 2),
        int(property_target['had_recent_violation'].sum()),
        round(property_target['had_recent_violation'].mean() * 100, 2),
        round(property_target.loc[property_target['had_violation'] == 1, 'violation_count'].mean(), 2),
    ]
})
target_summary.to_csv(PROCESSED_DIR / 'property_target_summary.csv', index=False)
print(target_summary.to_string(index=False))
print(f'\nSaved property-level modelling table -> {properties_with_violations_path}')
property_target[['PID', 'neighborhood', 'building_age_2025', 'TOTAL_VALUE', 'violation_count', 'had_violation']].head()


## Summary

The cleaning pipeline produces all of the project's analytical inputs:

- `data/processed/violations_clean.csv` - cleaned violation event records with three-layer neighborhood assignment.
- `data/processed/properties_clean.csv` - residential parcels with derived structural fields and neighborhood label.
- `data/processed/properties_with_violations.csv` - **property-level modelling table** with the `had_violation` target. **This is the file notebooks 03, 04, and 05 read.**
- `data/external/neighborhood_structural_features.csv` - small lookup of neighborhood housing-stock context.
- `data/processed/spatial_join_summary.csv` - reproducibility log of how many violations snapped within 50 m.
- `data/processed/property_target_summary.csv` - target balance and recency summary.

The spatial join is what makes property-level modelling possible: instead of 24 neighborhood rows, the model now sees ~150,000 residential parcels, of which a small fraction are labelled as violators by the BallTree haversine snap.

Notebook 03 explores patterns in `properties_with_violations.csv`. Notebook 04 builds the modelling features. Notebook 05 trains classification models for property-level violation risk.